## DataBase  creator vce format
This is an example what we want to do, from a text we generate the json file

In [ ]:
text='''
MLS-C01 
Number : 000-000 
Passing Score : 800 
Time Limit : 120 min 
File Version : 1.0 
  Exam A 
QUESTION 1 
A Machine Learning Specialist is working with multi ple data sources containing billions of records 
that need to be joined. What feature engineering an d model development approach should the 
Specialist take with a dataset this large? 
A. Use an Amazon SageMaker notebook for both feature  engineering and model development 
B. Use an Amazon SageMaker notebook for feature engi neering and Amazon ML for model development 
C. Use Amazon EMR for feature engineering and Amazon  SageMaker SDK for model development 
D. Use Amazon ML for both feature engineering and mo del development. 
Correct Answer: C
Section: (none) 
Explanation 
Explanation/Reference: 
Explanation: 
Amazon EMR is a service that can process large amou nts of data efficiently and cost-effectively. It 
can run distributed frameworks such as Apache Spark , which can perform feature engineering on big 
data. Amazon SageMaker SDK is a Python library that  can interact with Amazon SageMaker service to 
train and deploy machine learning models. It can al so use Amazon EMR as a data source for training 
data. Reference: 
Amazon EMR 
Amazon SageMaker SDK 
'''
questions=[
    {
        'question': 'A Machine Learning Specialist is working with multi ple data sources containing billions of records that need to be joined. What feature engineering an d model development approach should the Specialist take with a dataset this large? ',
        'options': [
            'A. Use an Amazon SageMaker notebook for both feature  engineering and model development ',
            'B. Use an Amazon SageMaker notebook for feature engi neering and Amazon ML for model development ',
            'C. Use Amazon EMR for feature engineering and Amazon  SageMaker SDK for model development ',
            'D. Use Amazon ML for both feature engineering and mo del development. '
        ],
        'correct': 'C. Use Amazon EMR for feature engineering and Amazon SageMaker SDK for model development',
        'explanation': 'Amazon EMR is a service that can process large amou nts of data efficiently and cost-effectively. It can run distributed frameworks such as Apache Spark , which can perform feature engineering on big data. Amazon SageMaker SDK is a Python library that  can interact with Amazon SageMaker service to train and deploy machine learning models. It can al so use Amazon EMR as a data source for training data',
        'references': 'Amazon SageMaker, Amazon EMR, Amazon ML'
    }

]

In [2]:
!pip install PyPDF2


[notice] A new release of pip is available: 24.1.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
provider="AWS"
format='vce'
exam='MLS-C01-January.pdf'

In [5]:
import PyPDF2
import os
# Get the current working directory
current_dir = os.getcwd()
# Define the file path by appending the file name to the current directory
pdf_file_path = os.path.join(current_dir, 'exams', provider,format ,exam)

def pdf_to_text(pdf_file_path):
    # Open the PDF file in read-binary mode
    with open(pdf_file_path, 'rb') as f:
        # Create a PyPDF2 reader object
        pdf = PyPDF2.PdfReader(f)
        # Initialize an empty string to store the text
        text = ''
        
        # Iterate through all the pages in the PDF
        for page in pdf.pages:
            # Extract the text from the page and append it to the text string
            text += page.extract_text()
        
        # Return the extracted text
        return text



In [37]:
# Call the pdf_to_text function and print the result
text = pdf_to_text(pdf_file_path)
print(text[:1000])


Amazon MLS-C01 : Practice Test 
Number : MLS-C01 
Passing Score : 800 
Time Limit : 120 min 
    
Exam Code: MLS-C01 
Title : AWS Certified Machine Learning Specialty 
Sections 
1. I&O&T Managed Services Traversing the Core 
2. Volume A 
3. Volume B Exam A 
QUESTION 1 
A Machine Learning Specialist is working with multi ple data sources containing billions of records 
that need to be joined. What feature engineering an d model development approach should the 
Specialist take with a dataset this large? 
A. Use an Amazon SageMaker notebook for both feature  engineering and model development 
B. Use an Amazon SageMaker notebook for feature engi neering and Amazon ML for model development 
C. Use Amazon EMR for feature engineering and Amazon  SageMaker SDK for model development 
D. Use Amazon ML for both feature engineering and mo del development. 
Correct Answer: C
Section: (none) 
Explanation 
Explanation/Reference: 
Amazon EMR is a service that can process large amou nts of data efficie

In [7]:
import re
import json
def extract_questions(text):
    questions = []
    current_question = None
    in_explanation = False
    in_references = False
    current_section = ''

    for line in text.split('\n'):
        line = line.strip()
        
        # Match question number
        if re.match(r'QUESTION \d+', line):
            if current_question:
                questions.append(current_question)
            current_question = {
                'question': '',
                'options': [],
                'correct': '',
                'explanation': '',
                'references': ''
            }
            current_section = 'question'
            current_question['question'] = ''
        # Match options
        elif re.match(r'[A-D]\.', line):
            current_section = 'options'
            current_question['options'].append(line)
        # Match correct answer
        elif line.startswith('Correct Answer:'):
            current_section = 'correct'
            correct_option_letter = line.replace('Correct Answer:', '').strip()
            for option in current_question['options']:
                if option.startswith(correct_option_letter + '.'):
                    current_question['correct'] = option
        # Match explanation
        elif line.startswith('Explanation'):
            current_section = 'explanation'
            current_question['explanation'] = line.replace('Explanation:', '').strip()
        # Match references
        elif line.startswith('Reference:'):
            current_section = 'references'
            current_question['references'] = line.replace('Reference:', '').strip()
        elif current_section == 'question':
            current_question['question'] += ' ' + line
        elif current_section == 'explanation':
            current_question['explanation'] += ' ' + line
        elif current_section == 'references':
            current_question['references'] += ' ' + line

    if current_question:
        questions.append(current_question)
        
    # Clean up extra spaces and format references
    for question in questions:
        question['question'] = question['question'].strip()
        question['explanation'] = question['explanation'].strip()
        question['references'] = ' '.join(question['references'].split()).strip()

    return questions

In [28]:
import re
import json

def extract_questions(text):
    questions = []
    current_question = None
    current_section = None
    current_option_index = -1 # To track current option being processed, -1 means no option started yet

    for line in text.split('\n'):
        line = line.strip()

        if re.match(r'QUESTION \d+', line):
            if current_question:
                if current_option_index != -1 and current_section == 'options': # if options section was started but no option is added yet, remove the empty last option
                    if not current_question['options'][-1]:
                        current_question['options'].pop()
                questions.append(current_question)

            current_question = {
                'question': '',
                'options': [],
                'correct': '',
                'explanation': '',
                'references': ''
            }
            current_section = 'question'
            current_question['question'] = line  # Start question with the QUESTION line itself
            current_option_index = -1 # reset option index for new question
        elif current_question is not None: # only process if we are inside a question block
            if current_section == 'question':
                if re.match(r'^[A-D]\.\s', line):
                    current_section = 'options'
                    current_question['options'].append([line]) # start new option with a list of lines, initialize with current line
                    current_option_index += 1 # move to next option index
                elif line.startswith('Correct Answer:'):
                    current_section = 'correct'
                    correct_option_letter = line.replace('Correct Answer:', '').strip()
                    correct_option_text = ""
                    for option_lines in current_question['options']: # iterate over list of lines for each option
                        option_text = "".join(option_lines) # join lines to form full option text for comparison
                        if option_text.startswith(correct_option_letter + '.'):
                            correct_option_text = option_text
                            break # Found the correct option, exit loop
                    current_question['correct'] = correct_option_text

                elif line.startswith('Explanation'):
                    current_section = 'explanation'
                    current_question['explanation'] = line.replace('Explanation:', '').strip()
                elif line.startswith('Explanation/Reference:'): # Handle combined explanation/reference
                    current_section = 'explanation'
                    current_question['explanation'] = line.replace('Explanation/Reference:', '').strip()
                elif line.startswith('Reference:'):
                    current_section = 'references'
                    current_question['references'] = line.replace('Reference:', '').strip()
                else:
                    current_question['question'] += ' ' + line # Append to question if not a new section

            elif current_section == 'options':
                if re.match(r'^[A-D]\.\s', line):
                    current_question['options'].append([line]) # start new option with a list of lines, initialize with current line
                    current_option_index += 1 # move to next option index
                elif line.startswith('Correct Answer:'):
                    current_section = 'correct'
                    correct_option_letter = line.replace('Correct Answer:', '').strip()
                    correct_option_text = ""
                    for option_lines in current_question['options']: # iterate over list of lines for each option
                        option_text = "".join(option_lines) # join lines to form full option text for comparison
                        if option_text.startswith(correct_option_letter + '.'):
                            correct_option_text = option_text
                            break # Found correct option, exit loop
                    current_question['correct'] = correct_option_text
                elif line.startswith('Explanation'):
                    current_section = 'explanation'
                    current_question['explanation'] = line.replace('Explanation:', '').strip()
                elif line.startswith('Explanation/Reference:'): # Handle combined explanation/reference
                    current_section = 'explanation'
                    current_question['explanation'] = line.replace('Explanation/Reference:', '').strip()
                elif line.startswith('Reference:'):
                    current_section = 'references'
                    current_question['references'] = line.replace('Reference:', '').strip()
                else: # it is continuation of current option
                    if current_option_index >= 0: # make sure there is option started
                        current_question['options'][current_option_index].append(line) # Append line to the last option


            elif current_section == 'correct':
                if line.startswith('Explanation'):
                    current_section = 'explanation'
                    current_question['explanation'] = line.replace('Explanation:', '').strip()
                elif line.startswith('Explanation/Reference:'): # Handle combined explanation/reference
                    current_section = 'explanation'
                    current_question['explanation'] = line.replace('Explanation/Reference:', '').strip()
                elif line.startswith('Reference:'):
                    current_section = 'references'
                    current_question['references'] = line.replace('Reference:', '').strip()
                else:
                    pass # Correct answer itself is already extracted, no more content for correct section

            elif current_section == 'explanation':
                if line.startswith('Reference:'):
                    current_section = 'references'
                    current_question['references'] = line.replace('Reference:', '').strip()
                else:
                    current_question['explanation'] += ' ' + line

            elif current_section == 'references':
                current_question['references'] += ' ' + line


    if current_question:
        if current_option_index != -1 and current_section == 'options': # if options section was started but no option is added yet, remove the empty last option
            if not current_question['options'][-1]:
                current_question['options'].pop()
        questions.append(current_question)

    # Clean up extra spaces and format references and options, and question
    for question_data in questions:
        question_data['question'] = question_data['question'].replace('QUESTION \d+', '').strip() # remove QUESTION N from question text itself
        question_data['explanation'] = question_data['explanation'].strip()
        question_data['references'] = ' '.join(question_data['references'].split()).strip()
        # Clean options: remove letter and number, just keep text, and join lines for each option
        cleaned_options = []
        for option_lines in question_data['options']:
            option_text = "".join(option_lines).strip() # join lines first
            cleaned_options.append(re.sub(r'^[A-D]\.\s*', '', option_text).strip()) # then remove letter and following dot and spaces
        question_data['options'] = cleaned_options
        if question_data['correct']:
            question_data['correct'] = re.sub(r'^[A-D]\.\s*', '', question_data['correct']).strip() # Clean correct answer too


    return questions

# Provided text
pdf_text = """
QUESTION 4 
A Machine Learning Specialist is using Amazon Sage Maker to host a model for a highly available 
customer-facing application. 
The Specialist has trained a new version of the mod el, validated it with historical data, and now 
wants to deploy it to production To limit any risk of a negative customer experience, the Specialist 
wants to be able to monitor the model and roll it b ack, if needed 
What is the SIMPLEST approach with the LEAST risk t o deploy the model and roll it back, if needed? 
A. Create a SageMaker endpoint and configuration for  the new model version. Redirect production 
traffic to the new endpoint by updating the client configuration. Revert traffic to the last version i f 
the model does not perform as expected. 
B. Create a SageMaker endpoint and configuration for  the new model version. Redirect production 
traffic to the new endpoint by using a load balance r Revert traffic to the last version if the model 
does not perform as expected. 
C. Update the existing SageMaker endpoint to use a n ew configuration that is weighted to send 5% 
of the traffic to the new variant. Revert traffic t o the last version by resetting the weights if the model 
does not perform as expected. 
D. Update the existing SageMaker endpoint to use a n ew configuration that is weighted to send 
100% of the traffic to the new variant Revert traff ic to the last version by resetting the weights if the 
model does not perform as expected. 
Correct Answer: C
Section: (none) 
Explanation 
Explanation/Reference: 
Updating the existing SageMaker endpoint to use a n ew configuration that is weighted to send 5% of 
the traffic to the new variant is the simplest appr oach with the least risk to deploy the model and ro ll 
it back, if needed. This is because SageMaker suppo rts A/B testing, which allows the Specialist to 
compare the performance of different model variants  by sending a portion of the traffic to each 
variant. The Specialist can monitor the metrics of each variant. This way, the Specialist
can deploy the model without affecting the customer experience and roll it back easily if
needed. References:
Amazon SageMaker
Deploying models to Amazon SageMaker hosting services
"""

questions = extract_questions(pdf_text)
print(json.dumps(questions, indent=4))

[
    {
        "question": "QUESTION 4 A Machine Learning Specialist is using Amazon Sage Maker to host a model for a highly available customer-facing application. The Specialist has trained a new version of the mod el, validated it with historical data, and now wants to deploy it to production To limit any risk of a negative customer experience, the Specialist wants to be able to monitor the model and roll it b ack, if needed What is the SIMPLEST approach with the LEAST risk t o deploy the model and roll it back, if needed?",
        "options": [
            "Create a SageMaker endpoint and configuration for\u00a0 the new model version. Redirect productiontraffic to the new endpoint by updating the client configuration. Revert traffic to the last version i fthe model does not perform as expected.",
            "Create a SageMaker endpoint and configuration for\u00a0 the new model version. Redirect productiontraffic to the new endpoint by using a load balance r Revert traffic to the l

In [41]:
questions = extract_questions(text)
print(json.dumps(questions[:4], indent=4))

[
    {
        "question": "QUESTION 1 A Machine Learning Specialist is working with multi ple data sources containing billions of records that need to be joined. What feature engineering an d model development approach should the Specialist take with a dataset this large?",
        "options": [
            "Use an Amazon SageMaker notebook for both feature  engineering and model development",
            "Use an Amazon SageMaker notebook for feature engi neering and Amazon ML for model development",
            "Use Amazon EMR for feature engineering and Amazon  SageMaker SDK for model development",
            "Use Amazon ML for both feature engineering and mo del development."
        ],
        "correct": "Use Amazon EMR for feature engineering and Amazon  SageMaker SDK for model development",
        "explanation": "Explanation Explanation/Reference: Amazon EMR is a service that can process large amou nts of data efficiently and cost-effectively. It can run distributed frameworks

In [12]:
type(questions[0])

dict

In [13]:
len(questions)

75

In [42]:
output=exam.replace(".pdf",".json")
output

'MLS-C01-January.json'

In [43]:
import os
import json
current_dir = os.getcwd()
folder_path = os.path.join(current_dir, 'exams', provider)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
file_path = os.path.join(folder_path, output)
with open(file_path, 'w') as f:
    json.dump(questions, f, indent=4)

In [93]:
def select_exam_(exam_name):
    current_dir = os.getcwd()
    folder_path = os.path.join(current_dir, 'exams', provider )
    file_path = os.path.join(folder_path, f'{exam_name}.json')

    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            questions = json.load(f)
        print("Loaded {} questions".format(len(questions)))    
        return questions
    else:
        return None

In [29]:
exam_name='GCP-ML-vA'
#exam_name='MLS-C01'

In [47]:
selected_questions = select_exam_(exam_name)

Loaded 75 questions


In [48]:
print(json.dumps(questions[:1], indent=4))

[
    {
        "question": "You are building an ML model to detect anomalies in  real-time sensor data. You will use Pub/Sub to han dle incoming requests. You want to store the results fo r analytics and visualization. How should you confi gure the pipeline?",
        "options": [
            "A. 1 = Dataflow, 2 = AI Platform, 3 = BigQuery",
            "B. 1 = DataProc, 2 = AutoML, 3 = Cloud Bigtable",
            "C. 1 = BigQuery, 2 = AutoML, 3 = Cloud Functions",
            "D. 1 = BigQuery, 2 = AI Platform, 3 = Cloud Storage"
        ],
        "correct": "A. 1 = Dataflow, 2 = AI Platform, 3 = BigQuery",
        "explanation": "Explanation/Reference: https://cloud.google.com/solutions/building-anomaly -detection-dataflow-bigqueryml-dlp",
        "references": ""
    }
]


In [37]:

import os
import json
# Function to load question sets from a directory
def load_question_sets(directory='questions'):
    question_sets = []
    for filename in os.listdir(directory):
        if filename.endswith(".json"):
            question_sets.append(filename[:-5])  # remove the .json extension
    return question_sets
exams = load_question_sets('exams/AWS')
print("question_sets:", exams)


question_sets: ['DOP-C02-v1', 'MLS-C01-v1', 'MLS-C01-v2', 'MLS-C01-v3', 'MLS-C01-v4', 'MLS-C01', 'SAA-C03-v1', 'SAA-C03-v2', 'SAP-C02-v1']


In [9]:
import os
import json

def load_question_sets(directory='questions'):
    question_sets = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".json"):
                question_sets.append(os.path.join( file)[:-5])  # remove the .json extension
    return question_sets

exams = load_question_sets('exams/')
print("question_sets:", exams)

question_sets: []


In [ ]:
GCP-ML-vA

In [49]:
provider="AWS"
selected_questions = select_exam_('MLS-C01')

Loaded 264 questions


In [50]:
provider="GOOGLE"
selected_questions = select_exam_('GCP-ML-vA')


Loaded 75 questions


In [51]:
selected_questions[0]

{'question': 'You are building an ML model to detect anomalies in  real-time sensor data. You will use Pub/Sub to han dle incoming requests. You want to store the results fo r analytics and visualization. How should you confi gure the pipeline?',
 'options': ['A. 1 = Dataflow, 2 = AI Platform, 3 = BigQuery',
  'B. 1 = DataProc, 2 = AutoML, 3 = Cloud Bigtable',
  'C. 1 = BigQuery, 2 = AutoML, 3 = Cloud Functions',
  'D. 1 = BigQuery, 2 = AI Platform, 3 = Cloud Storage'],
 'correct': 'A. 1 = Dataflow, 2 = AI Platform, 3 = BigQuery',
 'explanation': 'Explanation/Reference: https://cloud.google.com/solutions/building-anomaly -detection-dataflow-bigqueryml-dlp',
 'references': ''}

# Generation of all json files

In [31]:
import os

pdf_files = []
# Get the current working directory
current_dir = os.getcwd()
# Define the file path to search for PDF files
pdf_path = os.path.join(current_dir, 'exams', 'AWS', 'vce')
# Iterate through all files in the pdf_path directory
for filename in os.listdir(pdf_path):
    # Check if the file is a PDF file
    if filename.endswith(".pdf"):
        # Append the file name (with extension) to the pdf_files list
        pdf_files.append(filename)

print("PDF files:", pdf_files)


PDF files: ['MLS-C01-January.pdf', 'MLS-C01-June.pdf', 'MLS-C01-v2.pdf', 'SAA-C03-June.pdf', 'SAP-C02-June.pdf']


In [32]:
import os
# Get the current working directory
pdf_file_paths=[]
current_dir = os.getcwd()
directory = os.path.join(current_dir,"exams")
for root, dirs, files in os.walk(directory):
    for file in files:
        if file.endswith(".pdf"):
            #pdf_file_paths.append(file)  # remove the .json extension
            pdf_file_paths.append(os.path.join(root, file))  # include the full path

print("PDF files:", pdf_file_path)


PDF files: c:\Dropbox\23-GITHUB\Projects\AWS-Exam-Simulator\exams\AWS\vce\MLS-C01-January.pdf


In [33]:
pdf_file_paths

['c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\AWS\\vce\\MLS-C01-January.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\AWS\\vce\\MLS-C01-June.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\AWS\\vce\\MLS-C01-v2.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\AWS\\vce\\SAA-C03-June.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\AWS\\vce\\SAP-C02-June.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\GOOGLE\\vce\\GCP-CA.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\GOOGLE\\vce\\GCP-ML-vA.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\GOOGLE\\vce\\GCP-ML-vB.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\MICROSOFT\\vce\\AI-102.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\MICROSOFT\\vce\\AI-900-v1.pdf',
 'c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\MICROSOFT\\vce\\

In [34]:
pdf_file_paths=[pdf_file_paths[0]]

In [35]:
pdf_file_paths

['c:\\Dropbox\\23-GITHUB\\Projects\\AWS-Exam-Simulator\\exams\\AWS\\vce\\MLS-C01-January.pdf']

In [36]:

import os
import json

current_dir = os.getcwd()
directory_save = os.path.join(current_dir, "questions")

# Process each PDF file
for pdf_file_path in pdf_file_paths:
    text = pdf_to_text(pdf_file_path)
    questions = extract_questions(text)
    
    # Extract the filename of the pdf_file_path
    filename = os.path.basename(pdf_file_path)
    
    # Save the JSON file
    name = filename[:-4]  # remove the .pdf extension
    file_path = os.path.join(directory_save, f"{name}.json")
    
    with open(file_path, 'w') as f:
        json.dump(questions, f, indent=4)
    
    print(f"Processed {filename} and saved as {name}.json")


Processed MLS-C01-January.pdf and saved as MLS-C01-January.json


In [17]:
import os
import json
def select_exam_(exam_name):
    file_path = os.path.join(os.getcwd(), 'questions', f'{exam_name}.json')
    
    try:
        with open(file_path, 'r') as f:
            questions = json.load(f)
            print(f"Loaded {len(questions)} questions")
            return questions  # Ensure the questions are returned here

    except FileNotFoundError:
        print(f"File {file_path} not found.")
        return []  # Return an empty list to indicate no questions were found


In [168]:

selected_questions = select_exam_('MLS-C01-v0624')

Loaded 264 questions


In [172]:
selected_questions[0]

{'question': 'A Machine Learning Specialist is working with multi ple data sources containing billions of records that need to be joined. What feature engineering an d model development approach should the Specialist take with a dataset this large?',
 'options': ['A. Use an Amazon SageMaker notebook for both feature  engineering and model development',
  'B. Use an Amazon SageMaker notebook for feature engi neering and Amazon ML for model development',
  'C. Use Amazon EMR for feature engineering and Amazon  SageMaker SDK for model development',
  'D. Use Amazon ML for both feature engineering and mo del development.'],
 'correct': 'C. Use Amazon EMR for feature engineering and Amazon  SageMaker SDK for model development',
 'explanation': 'Amazon EMR is a service that can process large amou nts of data efficiently and cost-effectively. It can run distributed frameworks such as Apache Spark , which can perform feature engineering on big data. Amazon SageMaker SDK is a Python library tha